In [ ]:
import numpy as np 
import pandas as pd
import pickle 
import cv2
import matplotlib.pyplot as plt
import skimage
import umap
from scipy.spatial import Delaunay

In [ ]:
#load in non_empty_pixel_som_clusters
#load in segmentation_mask 
#load in cluster_color_dict 

In [ ]:
def create_clustered_pixel(x, y, labels, x_max, y_max):
    img_labeled = np.zeros((x_max+1, y_max+1)) * np.nan
    img_labeled[x, y] = labels
    return img_labeled 

In [ ]:
key = "" # sample identifier

In [ ]:
registration_path = r"" # path to registered images

In [ ]:
cell_clustering_path = r"" # path to cell phenotyping results

In [ ]:
pixel_clustering_path = r"" # path to pixel clustering results

In [ ]:
non_empty_pixels_som_clusters = pd.read_csv(pixel_clustering_path + '/non_empty_pixels_som_clusters.csv')

In [ ]:
som_cluster_weights = pd.read_csv(pixel_clustering_path + '/som_weights.csv').drop(columns=['Unnamed: 0'])

In [ ]:
with open(registration_path + '/roi_coord_dict.pickle', 'rb') as f: 
    roi = pickle.load(f)

In [ ]:
with open(pixel_clustering_path + '/cluster_color_dict.pickle', 'rb') as f: 
    cluster_color_dict = pickle.load(f)

In [ ]:
with open(cell_clustering_path + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f: 
    cell_segmentation_mask = pickle.load(f)

In [ ]:
cell_segmentation_mask_binary = np.zeros(cell_segmentation_mask.shape)
cell_segmentation_mask_binary[cell_segmentation_mask!=0] = 1
cell_segmentation_mask_binary = cv2.dilate(cv2.erode(cell_segmentation_mask_binary, np.ones((3, 3), np.uint8), iterations=1), np.ones((83, 83), np.uint8), iterations=1)

contours, _ = cv2.findContours(cell_segmentation_mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour = max(contours, key=cv2.contourArea)

contour_drawn = np.zeros((cell_segmentation_mask_binary.shape[0], cell_segmentation_mask_binary.shape[1]))
contour_drawn = cv2.drawContours(contour_drawn, [contour], -1, (255, 255, 255), -1)
contour_drawn = contour_drawn
plt.imshow(contour_drawn)

In [ ]:
target_clusters = [6, 7, 8, 9]
target_non_empty_pixels_som_clusters = non_empty_pixels_som_clusters[non_empty_pixels_som_clusters['pixel_som_cluster'].isin(target_clusters)]
x = target_non_empty_pixels_som_clusters.X.tolist()
y = target_non_empty_pixels_som_clusters.Y.tolist()
wsi_labeled = create_clustered_pixel(x, y, np.array(list(target_non_empty_pixels_som_clusters.pixel_som_cluster)), cell_segmentation_mask.shape[0]-1, cell_segmentation_mask.shape[1]-1)

wsi_labeled[wsi_labeled == 0] = np.nan
wsi_labeled[contour_drawn == 0] = np.nan

fig, ax = plt.subplots(1)
fig.set_figheight(8)
fig.set_figwidth(8)

k = 'C'
cmap = plt.cm.colors.ListedColormap(list(cluster_color_dict.values()))

ax.imshow(wsi_labeled[roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]], cmap=cmap, vmin=1, vmax=9)
ax.axis('off')

In [ ]:
import matplotlib as mpl
mpl.rcParams["font.size"] = 15
tile_size = 512
height, width = cell_segmentation_mask.shape

non_empty_pixels_som_clusters['tile_x'] = non_empty_pixels_som_clusters['X'] // tile_size
non_empty_pixels_som_clusters['tile_y'] = non_empty_pixels_som_clusters['Y'] // tile_size

tile_cluster_counts = (
    non_empty_pixels_som_clusters
    .groupby(['tile_y', 'tile_x', 'pixel_som_cluster'])
    .size()
    .reset_index(name='pixel_count')
)

pivot = tile_cluster_counts.pivot_table(
    index=['tile_y', 'tile_x'],
    columns='pixel_som_cluster',
    values='pixel_count',
    fill_value=0
)

pivot_fraction = pivot.div(pivot.sum(axis=1), axis=0)

mean_per_cluster = pivot_fraction.mean(axis=0)
std_per_cluster = pivot_fraction.std(axis=0)
num_tiles = len(pivot_fraction)
stderr_per_cluster = std_per_cluster / np.sqrt(num_tiles)

colors = [cluster_color_dict[c] for c in mean_per_cluster.index]

plt.figure(figsize=(6, 6))
ax = plt.gca()

ax.bar(
    mean_per_cluster.index,
    mean_per_cluster.values,
    xerr=stderr_per_cluster.values,
    capsize=5,
    color=colors, width=1
)

ax.set_xlabel('Cluster')
ax.set_ylabel('Fraction Pixels Assigned')
ax.set_xticks(np.arange(1, 17))
ax.tick_params(axis='x', labelrotation=30)

plt.show()


In [ ]:
W = np.array(som_cluster_weights)

In [ ]:
reducer = umap.UMAP(
    n_neighbors=8,
    min_dist=0.2,
    metric='euclidean',
    random_state=42
)
coords = reducer.fit_transform(W)  

In [ ]:
triangle = Delaunay(coords)
edges = set()

for simplex in triangle.simplices:
    i, j, k = simplex
    edges.add(tuple(sorted((i, j))))
    edges.add(tuple(sorted((j, k))))
    edges.add(tuple(sorted((i, k))))

plt.figure(figsize=(8, 8))

for i, j in edges:
    plt.plot(
        [coords[i, 0], coords[j, 0]],
        [coords[i, 1], coords[j, 1]],
        color='gray',
        alpha=0.5,
        linewidth=0.7
    )

colors = np.arange(len(coords))

plt.scatter(
    coords[:, 0],
    coords[:, 1],
    c=list(cluster_color_dict.values()),
    s=1000,
    edgecolors='black',
    linewidths=1.2,
    zorder=3
)
plt.axis('off')
plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()



#### Visualize tissue divided by grid

In [ ]:
contour_grid = np.ones(cell_segmentation_mask.shape, dtype=np.uint8) * 255
h, w = cell_segmentation_mask.shape
nx = 25
for i in range(1, nx):
    x = i * w // nx
    cv2.line(contour_grid, (x, 0), (x, h), 128, 25)
ny = 25
for i in range(1, ny):
    y = i * h // ny
    cv2.line(contour_grid, (0, y), (w, y), 128, 25)

contour_grid[contour_drawn == 0] = 255
contour_grid = cv2.drawContours(contour_grid, [contour], -1, color=128, thickness=50  
)


fig, ax = plt.subplots(figsize=(20, 25), dpi=100)
ax.imshow(skimage.color.gray2rgb(contour_grid), interpolation="nearest")
ax.axis("off")